In [ ]:
# Cell 1: Install required dependencies
# Pinned requests to 2.32.4 avoids breaking the native google-colab environment
!pip install --upgrade -q docling pypdf tqdm dataclasses-json
!pip install -q requests==2.32.4

In [ ]:
# Cell 2: Imports
import os
import re
import sys
import json
import time
import math
import logging
import hashlib
import shutil
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional, Tuple
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field, asdict

import requests
import pypdf
from tqdm import tqdm
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

In [ ]:
# Cell 3: Configuration
# Centralized configuration class following software engineering best practices.
@dataclass(frozen=True)
class AppConfig:
    # Directory paths
    BASE_DIR: Path = Path("project")
    METADATA_DIR: Path = BASE_DIR / "metadata"
    PDF_ORIGINAL_DIR: Path = BASE_DIR / "pdf" / "original"
    PDF_SPLIT_DIR: Path = BASE_DIR / "pdf" / "split"
    MARKDOWN_PARTS_DIR: Path = BASE_DIR / "markdown" / "parts"
    MARKDOWN_FINAL_DIR: Path = BASE_DIR / "markdown" / "final"
    LOGS_DIR: Path = BASE_DIR / "logs"

    # Files
    INPUT_JSON_NAME: str = "input.json"
    MANIFEST_NAME: str = "manifest.json"

    # Logs
    MAIN_LOG: str = "processing.log"
    COMPLETED_LOG: str = "completed.log"
    FAILED_LOG: str = "failed.log"

    # Network settings
    USER_AGENT: str = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    DOWNLOAD_TIMEOUT: int = 45  # seconds
    MAX_DOWNLOAD_RETRIES: int = 2
    BACKOFF_FACTOR: float = 2.0

    # Threading
    MAX_WORKERS: int = 4

    # Split Thresholds
    SPLIT_THRESHOLD_PAGES: int = 10
    CHUNK_SIZE_MEDIUM: int = 10   # For 11-50 pages
    CHUNK_SIZE_LARGE: int = 20    # For >50 pages

    # Docling Configuration
    DOCLING_THREADS: int = 2

# Instantiate global configuration object
config = AppConfig()
print("Configuration loaded.")

Configuration loaded.


In [ ]:
# Cell 4: Directory Creation
def initialize_directories(cfg: AppConfig) -> None:
    """Creates all pipeline directories safely if they do not exist."""
    directories = [
        cfg.BASE_DIR, cfg.METADATA_DIR, cfg.PDF_ORIGINAL_DIR,
        cfg.PDF_SPLIT_DIR, cfg.MARKDOWN_PARTS_DIR, cfg.MARKDOWN_FINAL_DIR,
        cfg.LOGS_DIR
    ]
    for directory in directories:
        directory.mkdir(parents=True, exist_ok=True)
    print("Pipeline directory hierarchy successfully initialized.")

initialize_directories(config)

Pipeline directory hierarchy successfully initialized.


In [ ]:
# Cell 5: Logging Setup
import logging.handlers

def setup_logging(cfg: AppConfig) -> Tuple[logging.Logger, logging.Logger, logging.Logger]:
    """Sets up separate flushed logging systems for global tracing, completions, and failures."""
    formatter = logging.Formatter('%(asctime)s | %(levelname)s | %(message)s')

    def _create_flushed_logger(name: str, log_file: Path) -> logging.Logger:
        logger = logging.getLogger(name)
        logger.setLevel(logging.INFO)
        # Prevent duplicate handlers if cell is re-run
        if logger.handlers:
            logger.handlers.clear()

        handler = logging.FileHandler(log_file, encoding='utf-8')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        return logger

    main_log = _create_flushed_logger("main_pipeline", cfg.LOGS_DIR / cfg.MAIN_LOG)
    comp_log = _create_flushed_logger("completed_pipeline", cfg.LOGS_DIR / cfg.COMPLETED_LOG)
    fail_log = _create_flushed_logger("failed_pipeline", cfg.LOGS_DIR / cfg.FAILED_LOG)

    # Attach console streamer to main logger for transparency
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    console_handler.setLevel(logging.WARNING)
    main_log.addHandler(console_handler)

    return main_log, comp_log, fail_log

logger, completed_logger, failed_logger = setup_logging(config)
print("Logging environment initialized successfully.")

Logging environment initialized successfully.


In [ ]:
# Cell 6: Dataclasses
@dataclass
class TaskItem:
    document_id: str
    title: str
    url: str
    declared_pages: int
    extracted_date: Optional[str]  # Format: YYYY-MM-DD
    priority: int                  # 1 or 2 (Priority 2 if no date or fallback)
    category: str = "Unknown"

@dataclass
class ManifestEntry:
    document_id: str
    title: str
    url: str
    date: Optional[str]
    pages: int
    priority: int
    status: str  # pending, downloading, downloaded, processing, completed, failed, skipped
    category: str = "Unknown"
    local_pdf: str = ""
    markdown_file: str = ""
    num_parts: int = 0
    last_processed: str = ""
    processing_time: float = 0.0
    error_message: str = ""
    pdf_sha256: str = ""
    markdown_sha256: str = ""

In [ ]:
# Cell 7: Manifest Manager
class ManifestManager:
    """Thread-safe and atomic operational manifest store tracking pipeline progress."""
    def __init__(self, cfg: AppConfig):
        self.path = cfg.METADATA_DIR / cfg.MANIFEST_NAME
        self.data: Dict[str, ManifestEntry] = {}
        self.load()

    def load(self) -> None:
        if self.path.exists():
            try:
                with open(self.path, 'r', encoding='utf-8') as f:
                    raw = json.load(f)
                    self.data = {k: ManifestEntry(**v) for k, v in raw.items()}
            except Exception as e:
                logger.error(f"Failed to read manifest, starting clean. Error: {str(e)}")
                self.data = {}
        else:
            self.data = {}

    def save_atomic(self) -> None:
        """Saves current tracking data using atomic writes to survive sudden disconnects."""
        temp_path = self.path.with_suffix(".tmp")
        try:
            with open(temp_path, 'w', encoding='utf-8') as f:
                serialized = {k: asdict(v) for k, v in self.data.items()}
                json.dump(serialized, f, indent=4, ensure_ascii=False)
                f.flush()
                os.fsync(f.fileno())
            temp_path.replace(self.path)
        except Exception as e:
            logger.error(f"Atomic manifest write failed: {str(e)}")

    def update_entry(self, entry: ManifestEntry) -> None:
        entry.last_processed = datetime.utcnow().isoformat() + "Z"
        self.data[entry.document_id] = entry
        self.save_atomic()

    def get_entry(self, doc_id: str) -> Optional[ManifestEntry]:
        return self.data.get(doc_id)

manifest_mgr = ManifestManager(config)
print("Manifest Manager initialized successfully.")

Manifest Manager initialized successfully.


In [ ]:
# Cell 8: Utility Functions
class PipelineUtils:
    @staticmethod
    def compute_sha256(text: str) -> str:
        """Generates the unique document identity sequence via string elements."""
        return hashlib.sha256(text.encode('utf-8')).hexdigest()

    @staticmethod
    def compute_file_sha256(file_path: Path) -> str:
        """Computes the file hash payload iteratively."""
        sha = hashlib.sha256()
        with open(file_path, 'rb') as f:
            while chunk := f.read(8192):
                sha.update(chunk)
        return sha.hexdigest()

    @staticmethod
    def sanitize_filename(filename: str, doc_id: str) -> str:
        """Clean string tokens into target system compatible files preserving hash differentiation."""
        name = Path(filename).stem
        ext = Path(filename).suffix
        sanitized = re.sub(r'[^a-zA-Z0-9_\-]', '_', name)
        # Handle collision and file limits truncation safety
        return f"{sanitized[:40]}_{doc_id[:8]}{ext}"

print("Pipeline Utilities initialized successfully.")

Pipeline Utilities initialized successfully.


In [ ]:
# Cell 9: Metadata Loader (Robust Version handling NSE Nested structures)
class MetadataLoader:
    """Ingests nested environment payload and isolates valid targets."""
    def __init__(self, cfg: AppConfig):
        self.path = cfg.METADATA_DIR / cfg.INPUT_JSON_NAME

    def extract_flat_entries(self) -> List[Dict[str, Any]]:
        if not self.path.exists():
            logger.error(f"Input JSON not found at {self.path}. Please upload it.")
            return []

        with open(self.path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        flat_entries = []

        # If fallback dummy data (list) was loaded, return it safely to prevent crashes
        if isinstance(raw_data, list):
            print("Notice: Detected a flat list JSON instead of the nested structure.")
            return raw_data

        # Traverse the nested NSE structure: Category -> Title -> [[URL, Pages], ...]
        if isinstance(raw_data, dict):
            for category, titles_dict in raw_data.items():
                if not isinstance(titles_dict, dict):
                    continue

                for title, url_list in titles_dict.items():
                    if not isinstance(url_list, list):
                        continue

                    for item in url_list:
                        if isinstance(item, list) and len(item) >= 1:
                            url = str(item[0]).strip()
                            try:
                                pages = int(item[1]) if len(item) > 1 else 0
                            except (ValueError, TypeError):
                                pages = 0

                            flat_entries.append({
                                "category": category,
                                "title": title,
                                "url": url,
                                "pages": pages
                            })

        print(f"Metadata Loader: Successfully flattened {len(flat_entries)} entries from the nested JSON.")
        return flat_entries

print("Metadata Loader initialized successfully.")

Metadata Loader initialized successfully.


In [ ]:
# Cell 10: Date Parser
class MultiFormatDateParser:
    """Advanced textual parser recovering timeline signatures across multiple format types."""

    MONTH_MAP = {
        'jan': 1, 'january': 1, 'feb': 2, 'february': 2, 'mar': 3, 'march': 3,
        'apr': 4, 'april': 4, 'may': 5, 'jun': 6, 'june': 6, 'jul': 7, 'july': 7,
        'aug': 8, 'august': 8, 'sep': 9, 'september': 9, 'oct': 10, 'october': 10,
        'nov': 11, 'november': 11, 'dec': 12, 'december': 12
    }

    @classmethod
    def parse_date(cls, text: str) -> Optional[datetime]:
        """Scans arbitrary strings to construct explicit datetime references."""
        if not text:
            return None
        text_clean = text.lower().strip()

        # 1. ISO Patterns & Numeric Blocks
        numeric_patterns = [
            r'(\d{4})[-_](\d{2})[-_](\d{2})',
            r'(\d{2})[-_](\d{2})[-_](\d{4})',
            r'\b(\d{8})\b'
        ]

        for pat in numeric_patterns[:2]:
            match = re.search(pat, text_clean)
            if match:
                g1, g2, g3 = match.groups()
                try:
                    if len(g1) == 4: return datetime(int(g1), int(g2), int(g3))
                    else: return datetime(int(g3), int(g2), int(g1))
                except ValueError: continue

        match = re.search(numeric_patterns[2], text_clean)
        if match:
            s = match.group(1)
            try:
                if s.startswith(('202', '201')): return datetime.strptime(s, "%Y%m%d")
                return datetime.strptime(s, "%d%m%Y")
            except ValueError: pass

        # 2. Textual Month extractions
        months_regex = '|'.join(cls.MONTH_MAP.keys())
        text_patterns = [
            rf'({months_regex})[-_\s]?(\d{{1,2}})[-_\s]?(\d{{4}})',
            rf'(\d{{1,2}})[-_\s]?({months_regex})[-_\s]?(\d{{4}})'
        ]

        for idx, pat in enumerate(text_patterns):
            match = re.search(pat, text_clean)
            if match:
                parts = match.groups()
                try:
                    if idx == 0: m_str, d_str, y_str = parts
                    else: d_str, m_str, y_str = parts
                    return datetime(int(y_str), cls.MONTH_MAP[m_str], int(d_str))
                except ValueError: continue

        return None

print("Multi-Format Date Parser initialized successfully.")

Multi-Format Date Parser initialized successfully.


In [ ]:
# Cell 11: Task Queue Builder
class TaskQueueBuilder:
    """Pre-sorts and organizes inputs into a robust execution queue before extraction."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg

    def build_queue(self, raw_items: List[Dict[str, Any]]) -> List[TaskItem]:
        processed_hashes = set()
        queue: List[TaskItem] = []

        for item in raw_items:
            url = item.get("url", "").strip()
            if not url:
                continue

            doc_id = PipelineUtils.compute_sha256(url)
            if doc_id in processed_hashes:
                continue
            processed_hashes.add(doc_id)

            title = item.get("title", "Untitled Document")
            declared_pages = int(item.get("pages", 0))

            search_str = f"{url} {title}"
            extracted_dt = MultiFormatDateParser.parse_date(search_str)

            priority = 2
            date_str = None
            if extracted_dt:
                date_str = extracted_dt.strftime("%Y-%m-%d")
                if datetime(2025, 7, 1) <= extracted_dt <= datetime(2026, 7, 1):
                    priority = 1

            queue.append(TaskItem(
                document_id=doc_id, title=title, url=url,
                declared_pages=declared_pages, extracted_date=date_str,
                priority=priority, category=item.get("category", "Filing")
            ))

        def sorting_key(x: TaskItem):
            p = x.declared_pages
            if p <= 4: bucket = 1
            elif p <= 10: bucket = 2
            elif p <= 50: bucket = 3
            else: bucket = 4
            return (x.priority, bucket, p)

        with_date = [t for t in queue if t.extracted_date is not None]
        without_date = [t for t in queue if t.extracted_date is None]

        with_date.sort(key=sorting_key)
        without_date.sort(key=sorting_key)

        return with_date + without_date

print("Task Queue Builder initialized successfully.")

Task Queue Builder initialized successfully.


In [ ]:
# Cell 12: Download Manager
class DownloadManager:
    """Manages streaming downloads with retry mechanics and error-handling capabilities."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": self.cfg.USER_AGENT})

    def download_document(self, task: TaskItem, dest_path: Path) -> bool:
        """Downloads files via resilient chunk streams, executing backoffs on failure."""
        retries = 0
        backoff = self.cfg.BACKOFF_FACTOR

        while retries <= self.cfg.MAX_DOWNLOAD_RETRIES:
            try:
                with self.session.get(task.url, stream=True, timeout=self.cfg.DOWNLOAD_TIMEOUT) as r:
                    r.raise_for_status()
                    with open(dest_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=65536):
                            if chunk:
                                f.write(chunk)
                return True
            except Exception as e:
                retries += 1
                if retries > self.cfg.MAX_DOWNLOAD_RETRIES:
                    logger.error(f"Failed download sequence for {task.url}. Exception: {str(e)}")
                    return False
                time.sleep(backoff)
                backoff *= 2
        return False

print("Download Manager initialized successfully.")

Download Manager initialized successfully.


In [ ]:
# Cell 13: PDF Validation
class PDFValidator:
    """Validates structural correctness of targeted documents before downstream execution."""
    @staticmethod
    def validate(file_path: Path) -> Tuple[bool, int]:
        """Validates PDF structural payload and extracts page dimensions cleanly."""
        if not file_path.exists() or file_path.stat().st_size == 0:
            return False, 0
        try:
            with open(file_path, 'rb') as f:
                reader = pypdf.PdfReader(f)
                pages = len(reader.pages)
                # Ensure the document contains readable pages
                if pages <= 0:
                    return False, 0
                return True, pages
        except Exception:
            return False, 0

print("PDF Validator initialized successfully.")

PDF Validator initialized successfully.


In [ ]:
# Cell 13: PDF Validation
class PDFValidator:
    """Validates structural correctness of targeted documents before downstream execution."""
    @staticmethod
    def validate(file_path: Path) -> Tuple[bool, int]:
        """Validates PDF structural payload and extracts page dimensions cleanly."""
        if not file_path.exists() or file_path.stat().st_size == 0:
            return False, 0
        try:
            with open(file_path, 'rb') as f:
                reader = pypdf.PdfReader(f)
                pages = len(reader.pages)
                # Ensure the document contains readable pages
                if pages <= 0:
                    return False, 0
                return True, pages
        except Exception:
            return False, 0

print("PDF Validator initialized successfully.")

PDF Validator initialized successfully.


In [ ]:
# Cell 14: PDF Splitting
class PDFSplitter:
    """Splits high-volume records into manageable document chunks for accelerated translation processing."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg

    def split_if_needed(self, file_path: Path, page_count: int) -> List[Path]:
        """Splits document fragments based on configurable depth profiles."""
        if page_count <= self.cfg.SPLIT_THRESHOLD_PAGES:
            return [file_path]

        chunk_size = self.cfg.CHUNK_SIZE_MEDIUM if page_count <= 50 else self.cfg.CHUNK_SIZE_LARGE
        split_paths: List[Path] = []

        try:
            reader = pypdf.PdfReader(file_path)
            total_pages = len(reader.pages)
            num_chunks = math.ceil(total_pages / chunk_size)

            for i in range(num_chunks):
                writer = pypdf.PdfWriter()
                start_page = i * chunk_size
                end_page = min(start_page + chunk_size, total_pages)

                for page_idx in range(start_page, end_page):
                    writer.add_page(reader.pages[page_idx])

                part_name = f"{file_path.stem}_part{i+1:02d}.pdf"
                part_path = self.cfg.PDF_SPLIT_DIR / part_name

                with open(part_path, 'wb') as out_f:
                    writer.write(out_f)
                split_paths.append(part_path)

            return split_paths
        except Exception as e:
            logger.error(f"Splitting structural error encountered on {file_path.name}: {str(e)}")
            return [file_path]

print("PDF Splitter initialized successfully.")

PDF Splitter initialized successfully.


In [ ]:
# Cell 15: Docling Converter
class DoclingConverterEngine:
    """Maintains instance layers matching internal structures with the high-performance structural converter engine."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg
        # Instantiating production conversion components safely using the updated API design
        options = PdfPipelineOptions()
        options.do_ocr = True  # Enable OCR fallback strategy for non-vector files
        options.do_table_structure = True

        # In Docling v2+, pipeline options must be passed via a format_options dictionary
        self.converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=options)
            }
        )

    def convert_file(self, pdf_path: Path, output_md_path: Path) -> bool:
        """Converts an individual document segment into standardized, clean markdown format."""
        try:
            # Native conversion extraction executed directly passing the Path instance
            result = self.converter.convert(pdf_path)
            markdown_content = result.document.export_to_markdown()

            with open(output_md_path, 'w', encoding='utf-8') as f:
                f.write(markdown_content)
            return True
        except Exception as e:
            logger.warning(f"Primary Docling translation failure execution instance on {pdf_path.name}: {str(e)}")
            return False

print("Docling Converter Engine initialized successfully.")

Docling Converter Engine initialized successfully.


In [ ]:
# Cell 16: Markdown Merger
class MarkdownMerger:
    """Assembles independent structural file nodes into highly context-preserved documents optimal for downstream RAG ingestion."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg

    def assemble_final_markdown(self, part_paths: List[Path], final_path: Path, manifest: ManifestEntry) -> None:
        """Appends segment parts cleanly, generating structured YAML frontiers across boundary segments."""
        front_matter = (
            "---\n"
            f"document_id: \"{manifest.document_id}\"\n"
            f"title: \"{manifest.title}\"\n"
            f"document_date: \"{manifest.date or 'Unknown'}\"\n"
            f"category: \"{manifest.category}\"\n"
            f"pages: {manifest.pages}\n"
            f"priority: {manifest.priority}\n"
            f"source_url: \"{manifest.url}\"\n"
            f"source_filename: \"{manifest.local_pdf}\"\n"
            f"downloaded_at: \"{manifest.last_processed}\"\n"
            f"processed_at: \"{datetime.utcnow().isoformat() + 'Z'}\"\n"
            "docling_version: \"latest\"\n"
            "company: \"Infosys\"\n"
            "exchange: \"NSE\"\n"
            "language: \"en\"\n"
            "---\n\n"
        )

        with open(final_path, 'w', encoding='utf-8') as out_f:
            out_f.write(front_matter)

            for idx, part in enumerate(part_paths):
                if not part.exists():
                    continue
                with open(part, 'r', encoding='utf-8') as in_f:
                    content = in_f.read()

                out_f.write(f"\n")
                out_f.write(content)
                out_f.write(f"\n\n\n")

print("Markdown Merger initialized successfully.")

Markdown Merger initialized successfully.


In [ ]:
# Cell 17: Resume Manager
class PipelineResumeEngine:
    """Manages transactional file states to verify tracking continuity across manual process resets."""
    @staticmethod
    def evaluate_execution_state(task: TaskItem, mgr: ManifestManager, cfg: AppConfig) -> Tuple[str, Optional[ManifestEntry]]:
        # Unsupportive target extensions isolated pre-execution
        url_lower = task.url.lower()
        if ".zip" in url_lower or ".rar" in url_lower or task.declared_pages < 0:
            entry = ManifestEntry(
                document_id=task.document_id, title=task.title, url=task.url,
                date=task.extracted_date, pages=task.declared_pages, priority=task.priority,
                status="skipped", category=task.category, error_message="Unsupported format structure payload generic signature."
            )
            return "skip", entry

        existing = mgr.get_entry(task.document_id)
        if existing:
            if existing.status == "completed":
                target_md = cfg.MARKDOWN_FINAL_DIR / existing.markdown_file
                if target_md.exists():
                    return "skip", existing
            if existing.status == "skipped":
                return "skip", existing

        return "process", existing

print("Pipeline Resume Engine initialized successfully.")

Pipeline Resume Engine initialized successfully.


In [ ]:
# Cell 18: Main Processing Pipeline Engine
class DocumentPipeline:
    """Core tracking orchestration framework executing pipeline tasks sequentially."""
    def __init__(self, cfg: AppConfig):
        self.cfg = cfg
        self.downloader = DownloadManager(cfg)
        self.splitter = PDFSplitter(cfg)
        self.converter = DoclingConverterEngine(cfg)
        self.merger = MarkdownMerger(cfg)

    def execute_task(self, task: TaskItem, entry: Optional[ManifestEntry]) -> str:
        start_time = time.time()
        if not entry:
            entry = ManifestEntry(
                document_id=task.document_id, title=task.title, url=task.url,
                date=task.extracted_date, pages=task.declared_pages,
                priority=task.priority, status="pending", category=task.category
            )

        # 1. Download
        orig_filename = PipelineUtils.sanitize_filename(os.path.basename(task.url) or "document.pdf", task.document_id)
        if not orig_filename.lower().endswith('.pdf'):
            orig_filename += ".pdf"

        pdf_dest = self.cfg.PDF_ORIGINAL_DIR / orig_filename
        entry.local_pdf = orig_filename

        if not pdf_dest.exists():
            entry.status = "downloading"
            manifest_mgr.update_entry(entry)
            success = self.downloader.download_document(task, pdf_dest)
            if not success:
                entry.status = "failed"
                entry.error_message = "Download timeout or server connection fault failure."
                manifest_mgr.update_entry(entry)
                failed_logger.info(f"{task.document_id} | Download Failed | {task.url}")
                return "failed"

        # 2. Validation
        valid, real_pages = PDFValidator.validate(pdf_dest)
        if not valid:
            logger.warning(f"File validation structural corruption hit on {orig_filename}. Executing full structural download clear re-attempt.")
            pdf_dest.unlink(missing_ok=True)
            success = self.downloader.download_document(task, pdf_dest)
            valid, real_pages = PDFValidator.validate(pdf_dest)
            if not valid:
                entry.status = "failed"
                entry.error_message = "Structural validation criteria verification failed on target data file payload."
                manifest_mgr.update_entry(entry)
                failed_logger.info(f"{task.document_id} | Validation Failed | {task.url}")
                return "failed"

        entry.pages = real_pages
        entry.pdf_sha256 = PipelineUtils.compute_file_sha256(pdf_dest)
        entry.status = "processing"
        manifest_mgr.update_entry(entry)

        # 3. Fragment Processing
        fragments = self.splitter.split_if_needed(pdf_dest, real_pages)
        entry.num_parts = len(fragments)
        part_md_paths: List[Path] = []

        conversion_failed = False
        for idx, frag in enumerate(fragments):
            part_md_name = f"{Path(orig_filename).stem}_part{idx+1:02d}.md"
            part_md_path = self.cfg.MARKDOWN_PARTS_DIR / part_md_name

            conv_ok = self.converter.convert_file(frag, part_md_path)
            if not conv_ok:
                conv_ok = self.converter.convert_file(frag, part_md_path)

            if not conv_ok:
                conversion_failed = True
                break
            part_md_paths.append(part_md_path)

        if conversion_failed:
            entry.status = "failed"
            entry.error_message = "Docling extraction conversion sequence failures encountered."
            manifest_mgr.update_entry(entry)
            failed_logger.info(f"{task.document_id} | Conversion Failed | {task.url}")
            return "failed"

        # 4. Assembly and Cleanup
        final_md_name = f"{Path(orig_filename).stem}.md"
        final_md_path = self.cfg.MARKDOWN_FINAL_DIR / final_md_name

        self.merger.assemble_final_markdown(part_md_paths, final_md_path, entry)

        if len(fragments) > 1:
            for frag in fragments:
                frag.unlink(missing_ok=True)
            for p_md in part_md_paths:
                p_md.unlink(missing_ok=True)

        entry.status = "completed"
        entry.markdown_file = final_md_name
        entry.markdown_sha256 = PipelineUtils.compute_file_sha256(final_md_path)
        entry.processing_time = time.time() - start_time
        manifest_mgr.update_entry(entry)

        completed_logger.info(f"{task.document_id} | Successfully Converted | {final_md_name}")
        return "completed"

print("Document Processing Pipeline engine initialized successfully.")

Document Processing Pipeline engine initialized successfully.


In [ ]:
# Cell 19: Execution Orchestrator Loop
def run_pipeline():
    """Main execution entry point utilizing standard pipeline structural instances."""
    print("Initializing Task Discovery and Compilation Sequence...")
    loader = MetadataLoader(config)
    builder = TaskQueueBuilder(config)

    raw_data = loader.extract_flat_entries()
    processing_queue = builder.build_queue(raw_data)

    pipeline = DocumentPipeline(config)

    print(f"Discovered {len(processing_queue)} distinct workload signatures. Initiating operational loop tracking sequence.\n")

    # Tracking monitor execution metrics setup
    progress_bar = tqdm(processing_queue, desc="Processing Queue", unit="doc")

    for task in progress_bar:
        progress_bar.set_postfix_str(f"ID: {task.document_id[:8]}... | Priority: {task.priority}")

        action, entry = PipelineResumeEngine.evaluate_execution_state(task, manifest_mgr, config)

        if action == "skip":
            if entry and entry.status == "skipped":
                manifest_mgr.update_entry(entry)
            continue

        try:
            pipeline.execute_task(task, entry)
        except Exception as e:
            logger.critical(f"Unhandled operational exception crash on task context identifier {task.document_id}: {str(e)}", exc_info=True)
            if entry:
                entry.status = "failed"
                entry.error_message = f"Unhandled pipeline crash: {str(e)}"
                manifest_mgr.update_entry(entry)

run_pipeline()

Initializing Task Discovery and Compilation Sequence...
Metadata Loader: Successfully flattened 502 entries from the nested JSON.
Discovered 502 distinct workload signatures. Initiating operational loop tracking sequence.



Processing Queue:   0%|          | 0/502 [00:00<?, ?doc/s, ID: d188a613... | Priority: 1]/tmp/ipykernel_8014/751749072.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry.last_processed = datetime.utcnow().isoformat() + "Z"
Processing Queue:   0%|          | 0/502 [00:00<?, ?doc/s, ID: 0105f974... | Priority: 1][INFO] 2026-07-07 10:59:23,018 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-07-07 10:59:23,023 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-07-07 10:59:23,027 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.9.1/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-07-07 10:59:23,648 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-07-07 10:59:24,247 [RapidOCR] download_file.py:95: Success

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

/tmp/ipykernel_8014/187751856.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"processed_at: \"{datetime.utcnow().isoformat() + 'Z'}\"\n"
/tmp/ipykernel_8014/751749072.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry.last_processed = datetime.utcnow().isoformat() + "Z"
INFO:completed_pipeline:0105f97475404293c34372f73a98684bff92a5883490f3041cb587f9a44523ed | Successfully Converted | INFY_01052026115302_2_SubmissionofApplic_0105f974.md
Processing Queue:   0%|          | 2/502 [00:19<1:20:03,  9.61s/doc, ID: 2d169c17... | Priority: 1]/tmp/ipykernel_8014/187751856.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a

2026-07-07 12:58:11,492 | WARNING | File validation structural corruption hit on Infosys_18032026153551_SEfiling_Reg30dis_951fb9f1.pdf. Executing full structural download clear re-attempt.


INFO:failed_pipeline:951fb9f12de64d6513ba5d66ab96b50988a1720d15c4acacfd1429a25e50d7d8 | Validation Failed | https://nsearchives.nseindia.com/corporate/Infosys_18032026153551_SEfiling_Reg30disclosure_18032026.pdf
Processing Queue:  19%|█▊        | 94/502 [1:58:49<02:37,  2.59doc/s, ID: b612c1ec... | Priority: 2]/tmp/ipykernel_8014/187751856.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"processed_at: \"{datetime.utcnow().isoformat() + 'Z'}\"\n"
/tmp/ipykernel_8014/751749072.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  entry.last_processed = datetime.utcnow().isoformat() + "Z"
INFO:completed_pipeline:b612c1ec6bd360b040b15a13d0fd7d90a243aadd13f8f7882fa49881a67d

In [ ]:
# Cell 20: Statistics & Final Summary Presentation Block
def generate_summary_report(cfg: AppConfig, mgr: ManifestManager):
    """Parses manifest execution payloads to construct clean operational reports."""
    mgr.load()
    entries = list(mgr.data.values())
    total = len(entries)

    pending = len([e for e in entries if e.status == "pending"])
    downloaded = len([e for e in entries if e.status in ["downloaded", "processing"]])
    completed = len([e for e in entries if e.status == "completed"])
    skipped = len([e for e in entries if e.status == "skipped"])
    failed = len([e for e in entries if e.status == "failed"])

    comp_times = [e.processing_time for e in entries if e.status == "completed" and e.processing_time > 0]
    avg_time = sum(comp_times) / len(comp_times) if comp_times else 0.0

    print("=" * 80)
    print("                       PIPELINE EXECUTION METRIC REPORT                ")
    print("=" * 80)
    print(f" Total Identified Track Records : {total}")
    print(f" Fully Processed & Converted    : {completed}")
    print(f" Skipped (Unsupported/Bounds)   : {skipped}")
    print(f" Failed Operational Entities    : {failed}")
    print(f" Unprocessed/Pending In Queue   : {pending + downloaded}")
    print(f" Average Processing Speed/Doc   : {avg_time:.2f} seconds")
    print("-" * 80)
    print(f" Target Clean Output Workspace  : {cfg.MARKDOWN_FINAL_DIR.resolve()}")
    print("=" * 80)

    if failed > 0:
        print("\n[!] CRITICAL TARGETED FAILURES IDENTIFIED REQUIRING RE-TRACKS:")
        failed_entries = [e for e in entries if e.status == "failed"]
        for idx, f_entry in enumerate(failed_entries[:15], 1):
            print(f"  {idx:02d}. ID: {f_entry.document_id[:8]}... | Title: {f_entry.title[:40]}... | Error: {f_entry.error_message}")
        if failed > 15:
            print(f"  ... and {failed - 15} other operational entity errors tracked inside {cfg.LOGS_DIR / cfg.FAILED_LOG}")

generate_summary_report(config, manifest_mgr)

                       PIPELINE EXECUTION METRIC REPORT                
 Total Identified Track Records : 504
 Fully Processed & Converted    : 492
 Skipped (Unsupported/Bounds)   : 10
 Failed Operational Entities    : 2
 Unprocessed/Pending In Queue   : 0
 Average Processing Speed/Doc   : 30.30 seconds
--------------------------------------------------------------------------------
 Target Clean Output Workspace  : /content/project/markdown/final

[!] CRITICAL TARGETED FAILURES IDENTIFIED REQUIRING RE-TRACKS:
  01. ID: ae6da43a... | Title: Sample Q1 Report 24... | Error: Download timeout or server connection fault failure.
  02. ID: 951fb9f1... | Title: Infosys Limited has informed the Exchang... | Error: Structural validation criteria verification failed on target data file payload.


In [ ]:
# Cell 21: Backup to Google Drive
import os
import shutil
from google.colab import drive

def backup_to_drive(source_dir: str = "project", backup_name: str = "NSE_Document_Pipeline_Backup"):
    """Zips the entire project directory and copies it safely to Google Drive."""
    print("Mounting Google Drive... (You will be prompted to grant access)")
    drive.mount('/content/drive')

    drive_base_path = "/content/drive/MyDrive"
    zip_filename = f"{backup_name}.zip"

    print(f"\nZipping the '{source_dir}' directory. This may take a moment for large datasets...")
    # Creates a zip archive of the project folder
    shutil.make_archive(backup_name, 'zip', source_dir)

    print("Transferring archive to Google Drive...")
    # Move the created zip file to the root of your Google Drive
    shutil.move(zip_filename, os.path.join(drive_base_path, zip_filename))

    print(f"\n✅ Backup Complete! ")
    print(f"You can now find '{zip_filename}' in the main folder of your Google Drive.")

backup_to_drive()

Mounting Google Drive... (You will be prompted to grant access)
Mounted at /content/drive

Zipping the 'project' directory. This may take a moment for large datasets...
Transferring archive to Google Drive...

✅ Backup Complete! 
You can now find 'NSE_Document_Pipeline_Backup.zip' in the main folder of your Google Drive.
